# 04 · Contract negotiation (Block D)

An offer is a proposal; an **agreement** is a contract both sides have signed. Data can only be transferred under an agreement.

## Concept: the DSP negotiation protocol

Negotiation is an **asynchronous** exchange of DSP messages between the two control planes:

```
consumer                                   provider
   │── ContractRequestMessage (offer) ───────►│  checks credentials against the contract policy
   │◄────────── ContractAgreementMessage ─────│  (or ContractNegotiationTerminationMessage)
   │── ContractAgreementVerificationMessage ─►│
   │◄─── ContractNegotiationEventMessage ─────│  "finalized"
```

Each side keeps a **ContractNegotiation** object with its own state machine. From the consumer's side the usual path is
`REQUESTING → REQUESTED → AGREED → VERIFYING → VERIFIED → FINALIZED`, or `TERMINATED` if the provider rejects the request.

Because it is asynchronous, the Management API answers **immediately** with an id. We then poll the state.

In [1]:
import dataspace as ds

data_asset = ds.recall("data_asset", produced_by="02_publishing.ipynb")
nobody_asset = ds.recall("nobody_asset", produced_by="02_publishing.ipynb")
consumer_api = ds.management_api("consumer")

## D1 · Step 1: get the offer for our `data` asset

**What:** the same catalog request as in notebook 03.  **Why:** the contract request must carry the provider's offer exactly as published, including its id.

In [2]:
catalog_request = {
    "@context": ds.EDC_CONTEXT,
    "@type": "CatalogRequest",
    "counterPartyAddress": ds.dsp_address("provider"),
    "counterPartyId": ds.did("provider"),
    "protocol": ds.DSP_PROTOCOL,
    "querySpec": {"limit": 500},
}
catalog = ds.post(f"{consumer_api}/catalog/request", catalog_request, expect={200}).json()

dataset = next(d for d in ds.as_list(catalog["dataset"]) if d["@id"] == data_asset)
offer = ds.as_list(dataset["hasPolicy"])[0]
print("offer id:", offer["@id"])

POST   /consumer/cp/management/v3/catalog/request  ->  200
offer id: Y29udHJhY3QtZ2luZy1kYXRhLTFkMGQ0ZQ==:Z2luZy1kYXRhLTFkMGQ0ZQ==:ZjNiYmU1M2YtNTFiNy00NWFiLTg3MjMtYjgwNmU3MjE5N2Ni


## D1 · Step 2: send the contract request

**What:** `POST /consumer/cp/management/v3/contractnegotiations`.

| Field | Meaning |
|---|---|
| `counterPartyAddress`, `protocol` | where and how our control plane sends the DSP `ContractRequestMessage` |
| `policy` | the offer, plus `assigner` (the provider's DID) and `target` (the asset). The provider checks it matches what it published |

`@context` is the Management API context, so the ODRL terms in the offer (`Offer`, `permission`, `eq`…) resolve to the ODRL namespace.

**Returns:** `IdResponse` with the negotiation id. At this point nothing has been agreed yet.

In [3]:
contract_request = {
    "@context": ds.MANAGEMENT_CONTEXT,
    "@type": "ContractRequest",
    "counterPartyAddress": ds.dsp_address("provider"),
    "protocol": ds.DSP_PROTOCOL,
    "policy": {**offer, "assigner": ds.did("provider"), "target": data_asset},
}
response = ds.post(f"{consumer_api}/contractnegotiations", contract_request, expect={200})
print(response.json())
negotiation_id = response.json()["@id"]

POST   /consumer/cp/management/v3/contractnegotiations  ->  200
{'@type': 'IdResponse', '@id': '63b504b7-fb96-437f-a571-3f68099fb858', 'createdAt': 1791554784557, '@context': {'dse-policy': 'https://w3id.org/dse/policy/', '@vocab': 'https://w3id.org/edc/v0.0.1/ns/', 'edc': 'https://w3id.org/edc/v0.0.1/ns/', 'odrl': 'http://www.w3.org/ns/odrl/2/'}}


## D1 · Step 3: wait for the outcome

**What:** poll `GET /v3/contractnegotiations/<id>/state`.  **Returns:** `{"state": "..."}`. The helper prints the states it saw.
The provider is meanwhile authenticating the consumer, requesting a presentation of its credentials and evaluating the contract policy in scope `contract.negotiation`.

In [4]:
def negotiation_state():
    return ds.get(f"{consumer_api}/contractnegotiations/{negotiation_id}/state", quiet=True).json()["state"]


final_state = ds.wait_for_state(negotiation_state, {"FINALIZED", "TERMINATED"})

   states: INITIAL -> REQUESTED -> AGREED -> VERIFIED -> FINALIZED


## D1 · Step 4: the negotiation and the agreement

**What:** `GET /v3/contractnegotiations/<id>`.  **Returns:** the consumer's negotiation record: `state`, `type` (`CONSUMER`), the counterparty,
`correlationId` (the provider's id for the same negotiation) and, once finalized, `contractAgreementId`.

In [5]:
negotiation = ds.get(f"{consumer_api}/contractnegotiations/{negotiation_id}").json()
for field in ["state", "type", "counterPartyId", "correlationId", "assetId", "contractAgreementId"]:
    print(f"   {field:20} {negotiation.get(field)}")
consumer_agreement_id = negotiation["contractAgreementId"]

GET    /consumer/cp/management/v3/contractnegotiations/63b504b7-fb96-437f-a571-3f68099fb858  ->  200
   state                FINALIZED
   type                 CONSUMER
   counterPartyId       did:web:provider-identityhub%3A8383:api:did
   correlationId        f3263c0e-a946-4fd7-95c2-838dc94b3f33
   assetId              ging-data-1d0d4e
   contractAgreementId  599d9a9f-a70c-43e1-9a6a-e34d115cf6a1


**What:** `GET /v3/contractagreements/<id>`.  **Returns:** the signed contract:

| Field | Meaning |
|---|---|
| `@id` | **this side's local id** for the agreement |
| `agreementId` | the id shared by both sides in the DSP messages |
| `assetId`, `providerId`, `consumerId` | what and between whom |
| `contractSigningDate` | when it was signed (epoch seconds); `inForceDate` constraints count from here |
| `policy` | an ODRL **Agreement**: the offer's permissions plus `assigner` (provider) and `assignee` (consumer) |

The provider stores the same agreement under a different local `@id`. Notebook 05 needs both.

In [6]:
agreement = ds.get(f"{consumer_api}/contractagreements/{consumer_agreement_id}").json()
print("consumer's local id :", agreement["@id"])
print("shared agreementId  :", agreement["agreementId"])
print("asset               :", agreement["assetId"])
print("provider / consumer :", agreement["providerId"], "/", agreement["consumerId"])
print("signed at           :", agreement["contractSigningDate"])
print("policy type         :", agreement["policy"]["@type"], "| assignee:", agreement["policy"]["odrl:assignee"])

ds.remember(consumer_agreement_id=consumer_agreement_id)

GET    /consumer/cp/management/v3/contractagreements/599d9a9f-a70c-43e1-9a6a-e34d115cf6a1  ->  200
consumer's local id : 599d9a9f-a70c-43e1-9a6a-e34d115cf6a1
shared agreementId  : c8cb959a-4cc4-496b-adbf-ab3ed91944d3
asset               : ging-data-1d0d4e
provider / consumer : did:web:provider-identityhub%3A8383:api:did / did:web:consumer-identityhub%3A8383:api:did
signed at           : 1791554785
policy type         : odrl:Agreement | assignee: did:web:consumer-identityhub%3A8383:api:did
   remembered consumer_agreement_id = 599d9a9f-a70c-43e1-9a6a-e34d115cf6a1


## D2 · A policy nobody satisfies

The `nobody` entry requires `name = nobody`. It is visible in the catalog (GenericClaim is not checked there), but the provider rejects the contract request.

From here on we use `ds.negotiate()`, which runs exactly steps 1–3 above and returns `(negotiation_id, final_state, agreement_id)`.

In [7]:
rejected_negotiation_id, final_state, _ = ds.negotiate(nobody_asset)
print("final state:", final_state)

POST   /consumer/cp/management/v3/contractnegotiations  ->  200


   states: INITIAL -> TERMINATED
final state: TERMINATED


**Why it was rejected:** a terminated negotiation keeps the provider's reason in `errorDetail`. It names the scope (`contract.negotiation`) and the constraint that failed.

In [8]:
rejected = ds.get(f"{consumer_api}/contractnegotiations/{rejected_negotiation_id}").json()
print(rejected["errorDetail"][:600])

GET    /consumer/cp/management/v3/contractnegotiations/b03ab1bd-84db-400a-92f2-4804a0343d64  ->  200
Failed to request contract to provider: {"@type":"ContractNegotiationError","code":"400","reason":["Contract offer is not valid: Policy in scope contract.negotiation not fulfilled: [Permission constraints: [Constraint 'https://w3id.org/dse/policy/GenericClaim.$.MembershipCredential.name' EQ 'nobody']]"],"@context":["https://w3id.org/dspace/2025/1/context.jsonld","https://w3id.org/edc/dspace/v0.0.1",{"dse-policy":"https://w3id.org/dse/policy/"}]}


## D5 · Delete the rejected negotiation

**What:** `DELETE /v3/contractnegotiations/<id>`.  **Returns:** 204, after which a GET returns 404. Useful for clearing failed attempts.

In [9]:
ds.delete(f"{consumer_api}/contractnegotiations/{rejected_negotiation_id}", expect={204})
ds.get(f"{consumer_api}/contractnegotiations/{rejected_negotiation_id}", expect={404})

DELETE /consumer/cp/management/v3/contractnegotiations/b03ab1bd-84db-400a-92f2-4804a0343d64  ->  204
GET    /consumer/cp/management/v3/contractnegotiations/b03ab1bd-84db-400a-92f2-4804a0343d64  ->  404


<Response [404]>

Next: **05_transfer.ipynb**